# Fixed points, orbits, and sliders

How does a system prompt affect the late-recurrence behaviour of individual tokens?
Compare Huginn and Retrofitted Llama on the same GSM8K questions, then inspect a
fixed PCA example, the most prevalent observed orbits, and the strongest slider.

Both modes use the two 128-recurrence runs and four prompt conditions below.
`mode="quick"` defaults to one seeded question and saves only in the run directory.
`mode="full"` covers all 1,319 GSM8K test questions and also updates the paper
figures; SLURM selects full mode. An explicit `limit` changes the sample count,
not the export policy.

The cohort needs only residual cosine similarities. PCA is computed for one
illustrative example; attention and residual entropy are collected only for the
selected orbit examples used in the stage figures.

In [ ]:
mode = "quick"
model_recurrences = [("raven", 128), ("retrofitted_llama", 128)]
prompt_names = ["no_prompt", "short_math", "long_persona", "long_persona_padded"]
limit = None  # Default: 1 (quick), all GSM8K test questions (full); 0 also means all.
seed = 42
device = "auto"
dtype = "auto"
output_dir = None

threshold = 0.01
lookback = 32
fixed_point_fraction = 0.9
late_recurrence_start = 64  # Zero-based: show recurrences 64-127 in stage profiles.
pca_run = ("raven", 128, "long_persona")
example_index = 18  # Fixed illustration; kept separate from prevalence if not sampled.
pca_token_count = 5

cache_dir = None  # Defaults to LOOPED_INTERNALS_CACHE_DIR or /data-gauss/hugbla/looped-internals-cache.
cache_mode = None  # Environment default; "reuse", "require" (no inference), or "off".
analysis_version = 1  # Bump when cached example or stage reductions change.

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
from dataclasses import asdict
from importlib.metadata import version

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset
from IPython.display import display
from sklearn.decomposition import PCA

from experiments.behavior_support import BehaviorRunConfig, capture_example
from experiments.cache import ExperimentCache
from experiments.support import (
    COLORMAPS,
    MODEL_LABELS,
    ModelSession,
    _set_seed,
    add_depth_colorbar,
    add_prelude_coda_connectors,
    configure_experiment,
    export_paper_figures,
    get_figsize,
    sample_indices,
    save_figure,
    save_results,
    write_tokenization,
)
from looped_internals.analysis import (
    get_colsum_concentrations,
    get_mixing_scores,
    get_residual_entropy,
    get_sink_scores,
)
from looped_internals.analysis.behaviors import _classify_series_without_last
from looped_internals.analysis.comparisons import COSINE_EPS
from looped_internals.models.tokenization import tokenizer_metadata

plt.style.use(ROOT / "plots" / "matlab.mplstyle")
model_recurrences = [tuple(pair) for pair in model_recurrences]
pca_run = tuple(pca_run)
if not model_recurrences or len(set(model_recurrences)) != len(model_recurrences):
    raise ValueError("Select at least one model-recurrence pair, without duplicates")
if not 3 <= lookback < min(steps for _, steps in model_recurrences):
    raise ValueError("lookback must be at least 3 and shorter than every rollout")
if pca_run[:2] not in model_recurrences or pca_run[2] not in prompt_names:
    raise ValueError("The PCA example must use one of the listed runs and prompts")

sample_limit = limit if limit is not None else (1 if mode == "quick" else 0)
configs = {
    pair: configure_experiment(
        experiment="limiting_behaviour",
        dataset="gsm8k",
        mode=mode,
        models=[pair[0]],
        num_steps=pair[1],
        limit=sample_limit,
        seed=seed,
        device=device,
        dtype=dtype,
        output_dir=output_dir,
    )
    for pair in model_recurrences
}
config = next(iter(configs.values()))
PROMPT_LABELS = {
    "no_prompt": "No system prompt",
    "short_math": "Short mathematical prompt",
    "long_persona": "Long persona",
    "long_persona_padded": "Long persona (padded)",
}
SLUGS = {"raven": "huginn", "retrofitted_llama": "retrofitted-llama"}
prompt_settings = {}
for name, recurrences in model_recurrences:
    for prompt in prompt_names:
        text_name = prompt.removesuffix("_padded")
        text = (
            ""
            if text_name == "no_prompt"
            else (ROOT / "experiments" / "prompts" / f"{text_name}.txt")
            .read_text()
            .strip()
        )
        prompt_settings[(name, recurrences, prompt)] = BehaviorRunConfig(
            SLUGS[name],
            text,
            prompt,
            n_recurrences=recurrences,
            replace_prompt_with_pad=prompt.endswith("_padded"),
            lookback=lookback,
            threshold=threshold,
        )

KINDS = ["FixedPoint", "Orbit", "Slider", "Unknown"]
STAGE_METRICS = {
    "sink_rates": "Sink rate",
    "mixing_scores": "Mixing score",
    "colsum_concentrations": "Column concentration",
    "residual_entropy": "Residual entropy",
}

result_cache = ExperimentCache(cache_dir, cache_mode)

In [ ]:
def cosine_histories(cache, entries, start):
    """Compare each question-region token with its own layer's final residual."""
    by_layer = {}
    for entry in entries:
        if entry.iteration is not None:
            by_layer.setdefault(entry.layer_index, []).append(entry)
    histories = {}
    for layer, applications in by_layer.items():
        final = cache[applications[-1].residual_key][0, start:].float()
        histories[layer] = torch.stack(
            [
                F.cosine_similarity(
                    cache[entry.residual_key][0, start:].float(),
                    final,
                    dim=-1,
                    eps=COSINE_EPS,
                ).clamp(-1, 1)
                for entry in applications
            ]
        )
    return histories


def classify_example(record):
    rows = []
    for layer, series in record["similarities"].items():
        behaviors = _classify_series_without_last(
            series[-lookback - 1 : -1],
            threshold,
            fixed_point_fraction,
        )
        for offset, behavior in enumerate(behaviors):
            rows.append(
                {
                    "model": record["model"],
                    "recurrences": record["recurrences"],
                    "prompt": record["prompt"],
                    "example_idx": record["example_idx"],
                    "layer_idx": layer,
                    "sequence_pos": record["question_start"] + offset,
                    "token_str": record["token_strs"][offset],
                    "behavior_type": type(behavior).__name__,
                    "orbit_frequency": getattr(behavior, "frequency", np.nan),
                    "orbit_amplitude": getattr(behavior, "amplitude", np.nan),
                    "slider_gradient": getattr(behavior, "gradient", np.nan),
                }
            )
    return pd.DataFrame(rows)

In [ ]:
def joint_pca(cache, entries, input_ids, start, tokenizer):
    """Fit one shared basis to all tokens and virtual layers, including prelude/coda."""
    residuals = torch.stack([cache[e.residual_key][0].float() for e in entries]).numpy()
    virtual_layers, tokens, width = residuals.shape
    pca = PCA(n_components=6, random_state=seed)
    coordinates = pca.fit_transform(residuals.reshape(-1, width)).reshape(
        virtual_layers, tokens, 6
    )
    positions = list(range(start, min(start + pca_token_count, tokens)))
    layers = np.asarray([e.layer_index for e in entries])
    core = sorted({e.layer_index for e in entries if e.iteration is not None})
    return {
        "coordinates": {
            layer: coordinates[layers == layer][:, positions] for layer in core
        },
        "layers": core,
        "positions": positions,
        "token_strs": [tokenizer.decode(int(input_ids[0, p])) for p in positions],
        "explained_variance_ratio": pca.explained_variance_ratio_,
    }

In [ ]:
def prevalence_tables(frame):
    examples, tokens = [], []
    for (name, recurrences, prompt), group in frame.groupby(
        ["model", "recurrences", "prompt"]
    ):
        labels = {
            "Model": f"{MODEL_LABELS[name]} ({recurrences} steps)",
            "System prompt": PROMPT_LABELS[prompt],
        }
        example_row, token_row = dict(labels), dict(labels)
        for kind in ["Non-Fixed-Point", "Orbit", "Slider", "Unknown"]:
            flags = (
                group.behavior_type.ne("FixedPoint")
                if kind == "Non-Fixed-Point"
                else group.behavior_type.eq(kind)
            )
            example_row[kind + " %"] = (
                100 * flags.groupby(group.example_idx).any().mean()
            )
            token_row[kind + " %"] = 100 * flags.mean()
        examples.append(example_row)
        tokens.append(token_row)
    return pd.DataFrame(examples), pd.DataFrame(tokens)


def conditional_cooccurrence(frame):
    identity = ["model", "recurrences", "prompt", "example_idx", "sequence_pos"]
    presence = (
        frame.assign(present=1)
        .pivot_table(
            index=identity,
            columns="behavior_type",
            values="present",
            aggfunc="max",
            fill_value=0,
        )
        .reindex(columns=KINDS, fill_value=0)
    )
    counts = presence.T @ presence
    totals = presence.sum()
    return counts.div(totals.where(totals > 0), axis="columns")


CASE_FIELDS = ["model", "recurrences", "prompt", "example_idx"]


def select_prevalent_orbit(frame, *, model=None):
    eligible = frame if model is None else frame[frame.model.eq(model)]
    orbits = eligible[eligible.behavior_type.eq("Orbit")]
    if orbits.empty:
        return None
    totals = eligible.groupby(CASE_FIELDS).size().rename("total_observations")
    ranking = (
        orbits.groupby(CASE_FIELDS)
        .agg(
            orbit_observations=("behavior_type", "size"),
            max_orbit_amplitude=("orbit_amplitude", "max"),
        )
        .join(totals)
        .reset_index()
    )
    ranking["orbit_prevalence"] = (
        ranking.orbit_observations / ranking.total_observations
    )
    ranking = ranking.sort_values(
        [
            "orbit_prevalence",
            "orbit_observations",
            "max_orbit_amplitude",
            *CASE_FIELDS,
        ],
        ascending=[False, False, False, True, True, True, True],
        kind="stable",
    )
    selected = ranking.iloc[0]
    mask = np.logical_and.reduce(
        [orbits[field].eq(selected[field]) for field in CASE_FIELDS]
    )
    case = (
        orbits[mask]
        .nlargest(1, "orbit_amplitude", keep="first")
        .iloc[0]
        .to_dict()
    )
    case.update(
        orbit_prevalence=float(selected.orbit_prevalence),
        orbit_observations=int(selected.orbit_observations),
        total_observations=int(selected.total_observations),
    )
    return case


def select_cases(frame):
    sliders = frame[frame.behavior_type.eq("Slider")]
    return {
        "most_prevalent_orbit": select_prevalent_orbit(frame),
        "retrofitted_most_prevalent_orbit": select_prevalent_orbit(
            frame, model="retrofitted_llama"
        ),
        "largest_slider": (
            None
            if sliders.empty
            else sliders.loc[sliders.slider_gradient.idxmax()].to_dict()
        ),
    }


def case_key(case):
    return (
        case["model"],
        int(case["recurrences"]),
        case["prompt"],
        int(case["example_idx"]),
    )

In [ ]:
def stage_curves(cache, entries, first_recurrence=0):
    """Compute only the stage measurements that appear in the selected figures."""
    entries = [
        e for e in entries if e.iteration is None or e.iteration >= first_recurrence
    ]
    layers = np.asarray([e.layer_index for e in entries])
    core = sorted({e.layer_index for e in entries if e.iteration is not None})
    by_layer = {f"L{layer}": np.flatnonzero(layers == layer) for layer in core}
    by_recurrence = {}
    if np.any(layers < min(core)):
        by_recurrence["Prelude"] = np.flatnonzero(layers < min(core))
    for iteration in sorted({e.iteration for e in entries if e.iteration is not None}):
        by_recurrence[f"R{iteration}"] = np.asarray(
            [i for i, e in enumerate(entries) if e.iteration == iteration]
        )
    if np.any(layers > max(core)):
        by_recurrence["Coda"] = np.flatnonzero(layers > max(core))
    curves = {field: [] for field in STAGE_METRICS}
    for entry in entries:
        attention = cache[entry.attn_key]
        curves["sink_rates"].append(
            (get_sink_scores(attention).float() >= 0.3).float().mean().item()
        )
        curves["mixing_scores"].append(
            get_mixing_scores(attention).float().mean().item()
        )
        curves["colsum_concentrations"].append(
            get_colsum_concentrations(attention).float().mean().item()
        )
        curves["residual_entropy"].append(
            get_residual_entropy(cache[entry.residual_key]).item()
        )
    return {
        "layers": layers,
        "virtual_layers": np.asarray([e.virtual_layer_index for e in entries]),
        "groups": {"layer": by_layer, "recurrence": by_recurrence},
        "curves": {field: np.asarray(values) for field, values in curves.items()},
    }


def orbit_spectrum(signal):
    trimmed = signal[-lookback - 1 : -1].float()
    n = len(trimmed)
    x = torch.arange(n, dtype=torch.float32)
    design = torch.stack([x, torch.ones(n)], dim=1)
    trend = design @ torch.linalg.lstsq(design, trimmed).solution
    detrended = trimmed - trend
    windowed = detrended * torch.hann_window(n, periodic=False)
    amplitudes = 4 * torch.fft.rfft(windowed).abs()[1:] / n
    frequencies = torch.arange(1, len(amplitudes) + 1).float() / n
    return trimmed, detrended, windowed, frequencies, amplitudes


def behavior_key(pair_config, name, prompt, index, **measurements):
    protocol = prompt_settings[(name, pair_config.num_steps, prompt)]
    return result_cache.key(
        pair_config,
        name,
        analysis_version=analysis_version,
        question=gsm8k[index]["question"],
        example_index=index,
        system_prompt=protocol.system_prompt,
        prompt=prompt,
        add_special_tokens=protocol.add_special_tokens,
        replace_prompt_with_pad=protocol.replace_prompt_with_pad,
        **measurements,
    )


def collect_behavior_example(session, pair, prompt, index, with_pca):
    bridge, structure = session.get()
    _set_seed((seed + index) % 2**32)
    input_ids, start, cache = capture_example(
        bridge,
        structure,
        gsm8k[index]["question"],
        prompt_settings[(*pair, prompt)],
        session.config.device,
    )
    entries = list(structure.virtual_layers(pair[1]))
    return {
        "model": pair[0],
        "recurrences": pair[1],
        "prompt": prompt,
        "example_idx": index,
        "input_ids": input_ids.cpu(),
        "question_start": start,
        "token_strs": [bridge.tokenizer.decode(int(t)) for t in input_ids[0, start:]],
        "similarities": cosine_histories(cache, entries, start),
        "pca": joint_pca(cache, entries, input_ids, start, bridge.tokenizer)
        if with_pca
        else None,
        "tokenization": {
            "bos_policy": "native",
            **tokenizer_metadata(bridge.tokenizer),
        },
    }


def collect_selected_stages(session, key, first):
    _name, recurrences, _prompt, index = key
    bridge, structure = session.get()
    _set_seed((seed + index) % 2**32)
    input_ids, start, cache = capture_example(
        bridge,
        structure,
        gsm8k[index]["question"],
        prompt_settings[key[:3]],
        session.config.device,
        attention=True,
    )
    record = torch.load(record_paths[key], weights_only=False)
    if (
        not torch.equal(input_ids.cpu(), record["input_ids"])
        or start != record["question_start"]
    ):
        raise ValueError("Selected-case inputs differ from the classified example")
    return stage_curves(cache, list(structure.virtual_layers(recurrences)), first)

## Generate and reduce

For each model, evaluate all four prompt conditions on the same questions.
Full mode visits the entire test split in dataset order; limited runs use a seeded
sample. This retains rare cases such as the no-prompt Llama orbit in example 721.
Keep only same-layer, final-reference cosine histories for tokens in the question
region. No attention, residual entropy, cross-layer comparisons, or successive
comparisons are computed for this cohort. Save each compact example once, then
release its activation cache.

The fixed PCA illustration uses Huginn with the long persona prompt on GSM8K test
example 18. If it is in the sample, reuse that forward pass; otherwise collect it
once without adding it to the prevalence statistics. Its shared PCA basis includes
the entire prompt and all virtual layers; only the first five question-region
tokens and recurrent layers are retained for plotting.

After classification, inspect the most prevalent orbit overall, the most prevalent
orbit in Retrofitted Llama, and the largest positive-gradient slider. Orbit
prevalence is the fraction of question token–layer observations classified as an
orbit within one model/prompt/example; ties prefer broader coverage, then amplitude.
Their signals already exist in the saved histories. Recapture only the distinct
orbit examples for stage
metrics, enabling entropy only there. For a distinct retrofitted-orbit example,
reduce only the displayed late recurrences plus prelude and coda. Missing categories are reported explicitly.

In [ ]:
gsm8k = load_dataset("openai/gsm8k", "main", split="test")
indices = (
    list(range(len(gsm8k)))
    if config.limit is None
    else sample_indices(len(gsm8k), config.limit, seed)
)
settings = asdict(config)
for field in ("models", "num_steps", "compute_residual_entropy"):
    del settings[field]
settings.update(
    output_dir=str(config.output_dir),
    model_recurrences=model_recurrences,
    prompts=[asdict(value) for value in prompt_settings.values()],
    indices=indices,
    threshold=threshold,
    lookback=lookback,
    fixed_point_fraction=fixed_point_fraction,
    late_recurrence_start=late_recurrence_start,
    pca_run=pca_run,
    example_index=example_index,
    pca_token_count=pca_token_count,
    cohort_metrics=["same_layer_final_residual_cosine"],
    selected_orbit_metrics=list(STAGE_METRICS),
    orbit_selection=(
        "token_layer_prevalence_then_count_then_amplitude"
    ),
    versions={
        name: version(name)
        for name in ("torch", "transformer-lens", "transformers", "scikit-learn")
    },
)
settings["cache"] = result_cache.settings
run_dir = save_results({}, settings, config.output_dir)

frames, record_paths = [], {}
for pair, pair_config in configs.items():
    name, recurrences = pair
    with ModelSession(name, pair_config, result_cache) as session:
        for prompt in prompt_names:
            selected = list(indices)
            if (*pair, prompt) == pca_run and example_index not in selected:
                selected.append(example_index)
            directory = run_dir / "similarities" / f"{name}-{recurrences}" / prompt
            directory.mkdir(parents=True)
            for index in selected:
                with_pca = (*pair, prompt) == pca_run and index == example_index
                key = behavior_key(
                    pair_config,
                    name,
                    prompt,
                    index,
                    pca_token_count=pca_token_count if with_pca else None,
                )
                with result_cache.entry("behavior_examples", key) as entry:
                    record = (
                        entry.load()
                        if entry.complete
                        else collect_behavior_example(
                            session, pair, prompt, index, with_pca
                        )
                    )
                    if not entry.complete:
                        entry.save(record)
                    frame = classify_example(record)
                    if index in indices:
                        frames.append(frame)
                    if with_pca:
                        pca_example, example_behaviors = record["pca"], frame
                    path = directory / f"{index}.pt"
                    entry.export(record, path)
                    record_paths[(*pair, prompt, index)] = path
                    write_tokenization(run_dir, name, record["tokenization"])
                    del record
            print(f"{MODEL_LABELS[name]}: {prompt} finished", flush=True)

behavior_df = pd.concat(frames, ignore_index=True)
del frames
behavior_df.to_parquet(run_dir / "behaviors.parquet", index=False)
example_prevalence, token_prevalence = prevalence_tables(behavior_df)
cooccurrence = conditional_cooccurrence(behavior_df)
cases = select_cases(behavior_df)

In [ ]:
# Reuse the cohort's exact signals, including the final self-comparison endpoint.
signals, selected_rows = {}, []
for label, case in cases.items():
    if case is None:
        selected_rows.append(
            {"Case": label, "Status": "Absent from the selected sample"}
        )
        continue
    record = torch.load(record_paths[case_key(case)], weights_only=False)
    offset = int(case["sequence_pos"]) - record["question_start"]
    signals[label] = record["similarities"][int(case["layer_idx"])][:, offset].clone()
    selected_rows.append({"Case": label, "Status": "Selected", **case})
    del record
selected_cases = pd.DataFrame(selected_rows)
selected_cases.to_csv(run_dir / "selected_cases.csv", index=False)

# Two selected orbits may refer to the same question: collect its stage metrics once.
stage_keys = {
    case_key(cases[label])
    for label in ("most_prevalent_orbit", "retrofitted_most_prevalent_orbit")
    if cases[label] is not None
}
stage_data = {}
for pair, pair_config in configs.items():
    keys = sorted(key for key in stage_keys if key[:2] == pair)
    with ModelSession(pair[0], pair_config, result_cache) as session:
        for key in keys:
            name, recurrences, prompt, index = key
            first = (
                0
                if key == case_key(cases["most_prevalent_orbit"])
                else late_recurrence_start
            )
            request = behavior_key(
                pair_config, name, prompt, index, first_recurrence=first
            )
            stage_data[key] = result_cache.run(
                "behavior_stages", request, collect_selected_stages, session, key, first
            )

torch.save(
    {
        "pca": pca_example,
        "example_behaviors": example_behaviors,
        "cases": cases,
        "signals": signals,
        "stages": stage_data,
    },
    run_dir / "overview.pt",
)
result_cache.write_manifest(run_dir)

## What the measurements mean

For layer $l$ and token $t$, compare each recurrence's residual $h_{r,l,t}$ with
that **same layer's** last residual $h_{R,l,t}$ using cosine similarity. The last
self-comparison is retained in the saved history but excluded from classification.
The classifier examines the preceding 32 recurrences, in this order:

| Category | Rule |
| --- | --- |
| Fixed point | At least `round(fixed_point_fraction × lookback)` similarities are at least `1 − threshold`. |
| Orbit | After linear detrending and a Hann window, the largest non-DC FFT amplitude is at least `threshold / 2`, with at least two cycles in the window. |
| Slider | Neither earlier rule applies, and the positive linear slope exceeds `threshold / lookback`. |
| Unknown | None of the rules applies. |

These are finite-rollout descriptions of a cosine signal, not proofs of a fixed
point or orbit in the full residual state. The FFT figures use **the same threshold
and window as the classifier**. Recurrence numbers, example indices, and layer
indices are zero-based.

| Summary | Counting unit |
| --- | --- |
| Example prevalence | A question counts once if the category occurs at any question-region token in any recurrent layer. |
| Token prevalence | Percentage of **token–layer observations**; longer questions contribute more observations. |
| Co-occurrence $P(Y\mid X)$ | A token counts once per model/prompt/example. Ask whether categories X and Y occur in any of its recurrent layers, not whether every layer agrees. |

Non-fixed-point includes Unknown. The question region starts at the recorded
prompt boundary and includes any trailing chat-template tokens, preserving the
prompt protocol. Padding replaces that prefix with the tokenizer's padding
(or EOS) token. Questions are evaluated without reference answers.

PCA uses one basis shared across tokens and layers. Each displayed trajectory is
then centred at its own mean; the red cross marks that mean. This differs from
fitting a separate PCA for each token. Stage curves are single-example values:
attention metrics average heads, while residual entropy uses the full residual
matrix spectrum. No uncertainty bands are drawn for these selected examples.

In [ ]:
def plot_cooccurrence():
    fig, ax = plt.subplots(
        figsize=get_figsize(False, (1, 1), height_per_subplot=2.6),
        layout="constrained",
    )
    image = ax.imshow(
        np.ma.masked_invalid(cooccurrence.to_numpy()), cmap="YlGnBu", vmin=0, vmax=1
    )
    for row in range(len(KINDS)):
        for col in range(len(KINDS)):
            value = cooccurrence.iloc[row, col]
            ax.text(
                col,
                row,
                "—" if pd.isna(value) else f"{value:.2f}",
                ha="center",
                va="center",
                color="white" if value > 0.6 else "black",
            )
    ax.set(
        xticks=range(len(KINDS)),
        xticklabels=KINDS,
        yticks=range(len(KINDS)),
        yticklabels=KINDS,
        xlabel="Given category X",
        ylabel="Category Y also occurs",
    )
    ax.grid(False)
    fig.colorbar(image, ax=ax, label="P(Y | X)")
    save_figure(fig, run_dir, "behavior_cooccurrence_heatmap", bbox_inches=None)


def plot_pca(filename, *, across_layers=False):
    layers = pca_example["layers"]
    panels = (
        [(layer, 0, 1) for layer in layers]
        if across_layers
        else [(layers[-1], 0, 1), (layers[-1], 2, 3), (layers[-1], 4, 5)]
    )
    tokens = pca_example["token_strs"]
    fig, axes = plt.subplots(
        len(tokens),
        len(panels),
        squeeze=False,
        layout="constrained",
        figsize=get_figsize(True, (len(tokens), len(panels)), height_per_subplot=1.25),
    )
    for row, token in enumerate(tokens):
        for col, (layer, a, b) in enumerate(panels):
            ax = axes[row, col]
            path = pca_example["coordinates"][layer][:, row][:, [a, b]]
            path = path - path.mean(axis=0)
            ax.plot(*path.T, color="darkblue", alpha=0.3, linewidth=0.8)
            points = ax.scatter(
                *path.T, c=np.arange(len(path)), cmap="viridis", s=5, rasterized=True
            )
            ax.scatter(0, 0, color="red", marker="x", s=12)
            bounds = np.maximum(np.abs(path).max(axis=0) * 1.2, 1e-6)
            ax.set(
                xlim=(-bounds[0], bounds[0]),
                ylim=(-bounds[1], bounds[1]),
                xlabel=f"PC {a + 1}",
                ylabel=f"{token!r}\nPC {b + 1}" if col == 0 else f"PC {b + 1}",
            )
            if row == 0:
                ax.set_title(
                    f"Layer {layer}" if across_layers else f"PC {a + 1}-{b + 1}"
                )
    add_depth_colorbar(fig, axes, "recurrence", mappable=points)
    save_figure(fig, run_dir, filename, bbox_inches=None)

In [ ]:
def plot_stages(label, filename, *, by="recurrence"):
    case = cases[label]
    fig, axes = plt.subplots(
        1,
        4,
        sharex=True,
        layout="constrained",
        figsize=get_figsize(True, (1, 4), height_per_subplot=1.5),
    )
    if case is None:
        for ax in axes:
            ax.set_axis_off()
        fig.text(0.5, 0.5, "No matching orbit in the selected sample", ha="center")
    else:
        data = stage_data[case_key(case)]
        groups = data["groups"][by]
        if by == "recurrence":
            groups = {
                key: indices
                for key, indices in groups.items()
                if not key.startswith("R") or int(key[1:]) >= late_recurrence_start
            }
        colored = [key for key in groups if key not in {"Prelude", "Coda"}]
        colors = dict(
            zip(
                colored,
                plt.colormaps[COLORMAPS[by + "s"]](np.linspace(0, 1, len(colored))),
                strict=True,
            )
        )
        colors.update(Prelude="orange", Coda="black")
        x = (
            data["virtual_layers"]
            if by == "layer"
            else data["layers"] / max(data["layers"].max(), 1) * 100
        )
        for ax, (field, ylabel) in zip(axes, STAGE_METRICS.items(), strict=True):
            lines = {}
            for key, positions in groups.items():
                (lines[key],) = ax.plot(
                    x[positions],
                    data["curves"][field][positions],
                    color=colors[key],
                    linewidth=1.5,
                )
            if by == "recurrence":
                # A late-only crop has no first-recurrence endpoint for the prelude.
                if "R0" not in lines:
                    lines.pop("Prelude", None)
                add_prelude_coda_connectors(ax, lines)
            ax.set(
                xlabel="Virtual layer" if by == "layer" else "Layer depth (%)",
                ylabel=ylabel,
            )
        add_depth_colorbar(fig, axes, by)
    save_figure(fig, run_dir, filename, bbox_inches=None)


def plot_orbit_diagnostics():
    fig, axes = plt.subplots(
        2,
        3,
        layout="constrained",
        figsize=get_figsize(True, (2, 3), height_per_subplot=1.8),
    )
    for row, label in enumerate(
        ("most_prevalent_orbit", "retrofitted_most_prevalent_orbit")
    ):
        if cases[label] is None:
            for ax in axes[row]:
                ax.set_axis_off()
            axes[row, 0].text(
                0.5,
                0.5,
                "No matching orbit",
                ha="center",
                transform=axes[row, 0].transAxes,
            )
            continue
        case, signal = cases[label], signals[label]
        trimmed, detrended, windowed, frequencies, amplitudes = orbit_spectrum(signal)
        start = len(signal) - len(trimmed) - 1
        axes[row, 0].plot(signal, color="gray", alpha=0.4)
        axes[row, 0].plot(np.arange(start, len(signal) - 1), trimmed, color="steelblue")
        axes[row, 0].axvspan(start, len(signal) - 2, alpha=0.12)
        axes[row, 0].set(
            title=f"{MODEL_LABELS[case['model']]} · example {int(case['example_idx'])}",
            xlabel="Recurrence",
            ylabel="Cosine with final state",
        )
        axes[row, 1].plot(detrended, label="Detrended")
        axes[row, 1].plot(windowed, label="Hann window", alpha=0.8)
        axes[row, 1].set(title="Lookback signal", xlabel="Recurrence within window")
        axes[row, 1].legend()
        axes[row, 2].bar(
            frequencies, amplitudes, width=0.8 / len(trimmed), color="steelblue"
        )
        axes[row, 2].axhline(
            threshold / 2, color="red", linestyle="--", label="Amplitude threshold"
        )
        axes[row, 2].axvline(
            frequencies[amplitudes.argmax()],
            color="orange",
            linestyle=":",
            label="Dominant frequency",
        )
        axes[row, 2].set(
            title="Orbit spectrum",
            xlabel="Cycles per recurrence",
            ylabel="Hann-corrected amplitude",
        )
        axes[row, 2].legend()
    save_figure(fig, run_dir, "orbit_algorithm_examples", bbox_inches=None)

## Prevalence across prompts

The two tables answer different questions: how often a question exhibits a
behaviour at all, and how much of the token–layer population exhibits it.
Both tables also have CSV and LaTeX exports in the run directory.

In [ ]:
example_prevalence.to_csv(run_dir / "behavior_existence.csv", index=False)
(run_dir / "behavior_existence.tex").write_text(
    example_prevalence.to_latex(
        index=False,
        float_format="%.2f",
        escape=True,
        label="tab:behavior_existence",
        caption="Percentage of questions exhibiting each behaviour in any recurrent layer.",
    )
)
display(example_prevalence.round(2))

In [ ]:
token_prevalence.to_csv(run_dir / "behavior_tokens.csv", index=False)
(run_dir / "behavior_tokens.tex").write_text(
    token_prevalence.to_latex(
        index=False,
        float_format="%.2f",
        escape=True,
        label="tab:behavior_tokens",
        caption="Percentage of question-region token-layer observations exhibiting each behaviour.",
    )
)
display(token_prevalence.round(2))

## Co-occurrence across recurrent layers

Each column conditions on category X appearing in at least one layer for a token;
each row gives the fraction also showing Y. A dash means X was absent from the
sample, so the conditional probability is undefined.

In [ ]:
cooccurrence.to_csv(run_dir / "behavior_cooccurrence.csv")
plot_cooccurrence()

## The fixed PCA illustration

Five rows follow the first five question-region tokens in Huginn's long-persona
example. First show PCs 1–2, 3–4, and 5–6 at the final recurrent layer; then compare
PCs 1–2 across every recurrent layer. Both grids reuse the same shared PCA basis.

In [ ]:
plot_pca("pca_final_layer")

In [ ]:
plot_pca("pca_intermediate", across_layers=True)

In [ ]:
display(example_behaviors)
example_behaviors.to_csv(run_dir / "pca_example_behaviors.csv", index=False)

## The most prevalent observed orbits and strongest slider

Selections are made over the sampled questions and all four prompts. Orbit cases
maximize the within-example fraction of orbiting token–layer observations; count
and then maximum amplitude break ties. The representative row is the largest-amplitude
orbit within that example. If the overall and retrofitted selections coincide, their
stage data is collected once and reused.

In [ ]:
display(selected_cases)

The first two full-width figures inspect the most prevalent orbit overall: follow each
recurrent layer through the rollout, then overlay its late-recurrence layer
profiles. The third figure shows late profiles for the most prevalent retrofitted orbit.
Orange and black are the prelude and coda; the recurrence colour scale covers only
the displayed late recurrences. Empty selections produce an explicit missing-case panel.

In [ ]:
plot_stages("most_prevalent_orbit", "largest_orbit_stability_in_recurrence", by="layer")

In [ ]:
plot_stages("most_prevalent_orbit", "largest_orbit_stability_in_percentage_depth")

In [ ]:
plot_stages("retrofitted_most_prevalent_orbit", "stability_in_percentage_depth_llama")

## Inspect the orbit detector

For each selected orbit, show the full cosine history, the detrended/windowed
lookback signal, and its Hann-corrected non-DC spectrum. The last recurrence's
self-comparison is excluded from the window. The selected slider's history is
saved alongside these signals in `overview.pt`; it requires no recapture.

In [ ]:
plot_orbit_diagnostics()

In [ ]:
print(run_dir)
paper_dir = export_paper_figures(config, run_dir, dataset=config.dataset)
if paper_dir is not None:
    print(f"Saved paper figures to {paper_dir}")